# Week 1


In [ ]:
!git clone https://github.com/UCDenver-ccp/CRAFT.git
%cd /content/CRAFT
!git pull


Cloning into 'CRAFT'...
remote: Enumerating objects: 17965, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 17965 (delta 152), reused 197 (delta 127), pack-reused 17730 (from 1)
Receiving objects: 100% (17965/17965), 258.71 MiB | 15.92 MiB/s, done.
Resolving deltas: 100% (15311/15311), done.
Updating files: 100% (3078/3078), done.
/content/CRAFT
Already up to date.


In [ ]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict

def parse_xml_annotations(xml_file, ontology_type):
    """Parse XML file and extract annotations with GO IDs and names"""
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Dictionary to store class mentions (GO IDs and names)
    class_mentions = {}
    for class_mention in root.findall('.//classMention'):
        mention_id = class_mention.get('id')
        mention_class = class_mention.find('mentionClass')
        if mention_class is not None:
            go_id = mention_class.get('id')
            go_name = mention_class.text
            class_mentions[mention_id] = (go_id, go_name)

    # List to store all annotations
    annotations = []

    # Process each annotation
    for annotation in root.findall('.//annotation'):
        mention = annotation.find('mention')
        if mention is None:
            continue

        mention_id = mention.get('id')
        span = annotation.find('span')
        if span is None:
            continue

        start = int(span.get('start'))
        end = int(span.get('end'))
        spanned_text = annotation.find('spannedText').text if annotation.find('spannedText') is not None else ""

        if mention_id in class_mentions:
            go_id, go_name = class_mentions[mention_id]
            annotations.append({
                'start': start,
                'end': end,
                'text': spanned_text,
                'go_id': go_id,
                'go_name': go_name,
                'ontology': ontology_type
            })

    # Get the article ID from the filename
    article_id = os.path.basename(xml_file).split('.')[0]

    return article_id, annotations

def process_ontology_directory(ontology_dir, ontology_type):
    """Process all XML files in a single ontology directory"""
    all_annotations = defaultdict(list)

    for xml_file in os.listdir(ontology_dir):
        if not xml_file.endswith('.xml'):
            continue

        xml_path = os.path.join(ontology_dir, xml_file)
        article_id, annotations = parse_xml_annotations(xml_path, ontology_type)
        all_annotations[article_id].extend(annotations)

    return all_annotations

def create_iob_files(all_annotations, output_dir, text_files_dir):
    """Create IOB format files for each document"""
    os.makedirs(output_dir, exist_ok=True)

    for article_id, annotations in all_annotations.items():
        # Sort annotations by start position
        annotations = sorted(annotations, key=lambda x: x['start'])

        # Create a dictionary to map character positions to annotations
        char_annotations = {}
        for ann in annotations:
            for pos in range(ann['start'], ann['end']):
                char_annotations[pos] = {
                    'go_id': ann['go_id'],
                    'go_name': ann['go_name'],
                    'ontology': ann['ontology']
                }

        # Read the original text file from the correct location
        text_file = os.path.join(text_files_dir, f"{article_id}.txt")
        try:
            with open(text_file, 'r', encoding='utf-8') as f:
                text = f.read()
        except FileNotFoundError:
            print(f"Warning: Text file not found at {text_file}")
            continue

        # Tokenize the text (simple whitespace tokenizer)
        tokens = []
        current_token = ""
        current_start = 0

        for i, char in enumerate(text):
            if char.isspace():
                if current_token:
                    tokens.append({
                        'token': current_token,
                        'start': current_start,
                        'end': i,
                        'go_id': None,
                        'go_name': None,
                        'ontology': None
                    })
                    current_token = ""
                continue

            if not current_token:
                current_start = i
            current_token += char

        # Add the last token if exists
        if current_token:
            tokens.append({
                'token': current_token,
                'start': current_start,
                'end': len(text),
                'go_id': None,
                'go_name': None,
                'ontology': None
            })

        # Assign annotations to tokens
        for token in tokens:
            # Check if any character in this token is annotated
            for pos in range(token['start'], token['end']):
                if pos in char_annotations:
                    token['go_id'] = char_annotations[pos]['go_id']
                    token['go_name'] = char_annotations[pos]['go_name']
                    token['ontology'] = char_annotations[pos]['ontology']
                    break

        # Determine IOB tags (keep 'O' for non-annotated tokens)
        prev_go_id = None
        for i, token in enumerate(tokens):
            if token['go_id']:
                if i == 0 or tokens[i-1]['go_id'] != token['go_id']:
                    token['iob'] = f"B-{token['go_id']}"
                else:
                    token['iob'] = f"I-{token['go_id']}"
                prev_go_id = token['go_id']
            else:
                token['iob'] = "O"
                prev_go_id = None

        # Write to IOB file
        output_file = os.path.join(output_dir, f"{article_id}.iob")
        with open(output_file, 'w', encoding='utf-8') as f:
            # Write header
            f.write(f"# Article ID: {article_id}\n")
            f.write("# Format: Token\tIOB_Tag\tGO_ID\tGO_Name\tOntology\n\n")

            # Write tokens
            for token in tokens:
                f.write(f"{token['token']}\t{token['iob']}\t")
                # Write empty string instead of 'O' for these fields
                f.write(f"{token['go_id'] if token['go_id'] else ''}\t")
                f.write(f"{token['go_name'] if token['go_name'] else ''}\t")
                f.write(f"{token['ontology'] if token['ontology'] else ''}\n")

def main():
    # Define paths (adjust these according to your actual paths)
    base_dir = "/content/CRAFT"

    ontology_dirs = {
        'GO_BP': os.path.join(base_dir, 'concept-annotation', 'GO_BP', 'GO_BP', 'knowtator'),
        'GO_CC': os.path.join(base_dir, 'concept-annotation', 'GO_CC', 'GO_CC', 'knowtator'),
        'GO_MF': os.path.join(base_dir, 'concept-annotation', 'GO_MF', 'GO_MF', 'knowtator')
    }

    text_files_dir = os.path.join(base_dir, 'articles', 'txt')
    output_dir = os.path.join(base_dir, 'iob_output')

    # Process all ontology directories and combine annotations
    combined_annotations = defaultdict(list)
    for ontology_type, ontology_dir in ontology_dirs.items():
        print(f"Processing {ontology_type} files...")
        annotations = process_ontology_directory(ontology_dir, ontology_type)
        for article_id, anns in annotations.items():
            combined_annotations[article_id].extend(anns)

    # Create IOB files
    create_iob_files(combined_annotations, output_dir, text_files_dir)
    print(f"Processing complete. IOB files saved to {output_dir}")

if __name__ == "__main__":
    main()

Processing GO_BP files...
Processing GO_CC files...
Processing GO_MF files...
Processing complete. IOB files saved to /content/CRAFT/iob_output


In [ ]:
import os
import json

def process_iob_files(iob_dir):
    """
    Process ALL tokens from IOB files exactly as they appear,
    but remove trailing periods and commas from tokens.
    """
    result_dict = {}
    token_counts = {}

    for filename in os.listdir(iob_dir):
        if not filename.endswith('.iob'):
            continue

        doc_id = os.path.splitext(filename)[0]
        filepath = os.path.join(iob_dir, filename)

        token_pairs = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue

                parts = line.split('\t')
                if len(parts) >= 2:
                    token = parts[0].rstrip(".,")  # 🔹 Remove trailing punctuation
                    iob_tag = parts[1]
                    token_pairs.append((token, iob_tag))

        result_dict[doc_id] = token_pairs
        token_counts[doc_id] = len(token_pairs)

    return result_dict, token_counts

def main():
    iob_dir = "/content/CRAFT/iob_output"

    # Process all files
    iob_data, token_counts = process_iob_files(iob_dir)

    # 1. Save complete JSON output
    output_file = "COMPLETE_iob_output.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(iob_data, f, indent=4, ensure_ascii=False)

    # 2. Save verification report
    verification_file = "TOKEN_COUNT_VERIFICATION.txt"
    with open(verification_file, 'w', encoding='utf-8') as f:
        f.write("COMPLETE TOKEN PROCESSING VERIFICATION\n")
        f.write("="*50 + "\n\n")
        f.write(f"Total documents processed: {len(iob_data)}\n\n")

        doc_id = "11319941"
        if doc_id in iob_data:
            f.write(f"Document {doc_id}:\n")
            f.write(f"Total tokens processed: {len(iob_data[doc_id])}\n")
            f.write(f"First 5 tokens: {iob_data[doc_id][:5]}\n")
            f.write(f"Last 5 tokens: {iob_data[doc_id][-5:]}\n\n")

        f.write("ALL DOCUMENTS TOKEN COUNTS:\n")
        f.write("="*30 + "\n")
        for doc_id, count in token_counts.items():
            f.write(f"{doc_id}: {count} tokens\n")

    print(f"COMPLETE token output saved to {output_file}")
    print(f"Verification report saved to {verification_file}")

    # Print sample from document 11319941
    if "11319941" in iob_data:
        print("\nSample from document 11319941:")
        print(f"Total tokens: {len(iob_data['11319941'])}")
        print("First 10 tokens:")
        for i, (token, tag) in enumerate(iob_data['11319941'][:10]):
            print(f"{i+1:>3}. {token:<20} {tag}")
        print("\nLast 10 tokens:")
        for i, (token, tag) in enumerate(iob_data['11319941'][-10:]):
            print(f"{len(iob_data['11319941'])-9+i:>3}. {token:<20} {tag}")

if __name__ == "__main__":
    main()


COMPLETE token output saved to COMPLETE_iob_output.json
Verification report saved to TOKEN_COUNT_VERIFICATION.txt

Sample from document 11319941:
Total tokens: 5425
First 10 tokens:
  1. Complex              O
  2. trait                O
  3. analysis             O
  4. of                   O
  5. the                  O
  6. mouse                O
  7. striatum:            O
  8. independent          O
  9. QTLs                 O
 10. modulate             B-GO:0065007

Last 10 tokens:
5416. We                   O
5417. thank                O
5418. Richelle             O
5419. Strom                O
5420. for                  O
5421. generating           O
5422. the                  O
5423. F2                   O
5424. intercross           O
5425. mice                 O


# Week 2


In [79]:
import re
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from collections import defaultdict, Counter
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from random import sample

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load BioBERT
bert_model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
biobert = AutoModel.from_pretrained(bert_model_name).to(device)

# Load IOB data
with open("COMPLETE_iob_output.json", "r") as f:
    raw_data = json.load(f)

# Enhanced tag vocabulary with strict IOB enforcement
tag2idx = defaultdict(lambda: len(tag2idx))
tag2idx["<PAD>"] = 0
standalone_entity_map = {}
stopwords = {'of', 'the', 'and', 'in', 'to', 'a', 'an', 'for', 'on', 'with', 'as', 'at', 'by'}

for _, token_pairs in raw_data.items():
    for token, tag in token_pairs:
        _ = tag2idx[tag]
        if tag.startswith("B-"):
            norm_token = token.lower()
            if norm_token not in standalone_entity_map:
                standalone_entity_map[norm_token] = tag[2:]

idx2tag = {v: k for k, v in tag2idx.items()}
pad_tag_id = tag2idx["<PAD>"]

# Chunking function
def chunk_tokens(token_pairs, max_len=128):
    chunks = []
    i = 0
    while i < len(token_pairs):
        end = i + max_len
        while end < len(token_pairs) and token_pairs[end][1].startswith("I-"):
            end += 1

        chunk = token_pairs[i:end]

        clean_chunk = [(re.sub(r'[^a-zA-Z0-9-]', '', token), tag)
                      for token, tag in chunk if re.sub(r'[^a-zA-Z0-9-]', '', token)]

        if clean_chunk:
            chunks.append(clean_chunk)

        # Add standalone B-tags only if they don't create duplicates
        if (token_pairs[i][1].startswith("B-") and
            (i+1 >= len(token_pairs) or not token_pairs[i+1][1].startswith("I-"))):
            clean_token = re.sub(r'[^a-zA-Z0-9-]', '', token_pairs[i][0])
            if clean_token and not any(clean_token in c[0] for c in chunks):
                chunks.append([(clean_token, token_pairs[i][1])])

        i = end

    return chunks

# Modified encoding process
encoded_data = []
for _, token_pairs in tqdm(raw_data.items(), desc="Processing data"):
    chunks = chunk_tokens(token_pairs, max_len=128)

    for chunk in chunks:
        words, tags = zip(*chunk)

        # Encode with dynamic padding
        encoded = tokenizer(
            list(words),
            is_split_into_words=True,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=128
        )

        word_ids = encoded.word_ids(batch_index=0)
        aligned_tags = []

        prev_word_idx = None
        for idx in word_ids:
            if idx is None:
                aligned_tags.append(pad_tag_id)
            elif idx != prev_word_idx:
                aligned_tags.append(tag2idx[tags[idx]])
            else:
                aligned_tags.append(tag2idx[tags[idx]] if tags[idx].startswith("I-") else pad_tag_id)
            prev_word_idx = idx

        # Dynamic padding implementation
        pad_len = 128 - len(aligned_tags)
        if pad_len > 0:
            aligned_tags = aligned_tags + [pad_tag_id] * pad_len
            encoded["input_ids"] = torch.cat([
                encoded["input_ids"],
                torch.full((1, pad_len), tokenizer.pad_token_id)
            ], dim=1)
            encoded["attention_mask"] = torch.cat([
                encoded["attention_mask"],
                torch.zeros((1, pad_len), dtype=torch.long)
            ], dim=1)

        encoded_data.append((
            encoded["input_ids"].squeeze(0).cpu(),
            encoded["attention_mask"].squeeze(0).cpu(),
            torch.tensor(aligned_tags)
        ))

# Print chunking statistics
print("\n=== Chunking Stats ===")
print(f"Total chunks generated: {len(encoded_data)}")
avg_chunk_size = sum(len(x[0]) for x in encoded_data)/len(encoded_data)
print(f"Average chunk size: {avg_chunk_size:.1f} tokens")

# Add tag distribution analysis
tag_counts = Counter()
for _, _, tags in encoded_data:
    tag_counts.update(tag.item() for tag in tags if tag != pad_tag_id)

# Entity density per chunk
entity_chunks = sum(1 for item in encoded_data
                   if any(idx2tag[t.item()].startswith(('B-', 'I-'))
                         for t in item[2] if t.item() != pad_tag_id))

# ===== DEDUPLICATION LOGIC =====
def ensure_no_duplicate_chunks(encoded_data):
    text_to_chunk = defaultdict(list)

    for item in encoded_data:
        input_ids = item[0]
        non_pad_ids = [i for i in input_ids.tolist() if i != tokenizer.pad_token_id]
        text_key = tuple(non_pad_ids)
        text_to_chunk[text_key].append(item)

    deduped_data = []
    for chunks in text_to_chunk.values():
        chunk_with_entity = next(
            (c for c in chunks if any(idx2tag[t.item()].startswith(('B-', 'I-'))
            for t in c[2] if t.item() != pad_tag_id)), chunks[0])
        deduped_data.append(chunk_with_entity)

    return deduped_data

print("\n=== Deduplication Check ===")
initial_count = len(encoded_data)
encoded_data = ensure_no_duplicate_chunks(encoded_data)
removed = initial_count - len(encoded_data)

final_texts = set()
for item in encoded_data:
    non_pad_ids = tuple(i for i in item[0].tolist() if i != tokenizer.pad_token_id)
    assert non_pad_ids not in final_texts, "Duplicate found!"
    final_texts.add(non_pad_ids)

# Calculate approximate number of tokens in encoded_data
total_tokens_approx = sum(len(x[0]) for x in encoded_data)

# Calculate target sample size
target_token_count = 500000
sample_ratio = target_token_count / total_tokens_approx if total_tokens_approx > 0 else 1.0
sample_size = int(len(encoded_data) * sample_ratio)

if len(encoded_data) > 0:
    encoded_data = sample(encoded_data, min(sample_size, len(encoded_data)))
unique_data = {}
for item in encoded_data:
    data_key = (tuple(item[0].tolist()), tuple(item[2].tolist()))
    if data_key not in unique_data:
        unique_data[data_key] = item
encoded_data = list(unique_data.values())
print(f"After final deduplication: {len(encoded_data)} chunks")

# Stratified 80-20 split
def contains_entity(tags):
    return any(idx2tag[t].startswith(('B-', 'I-')) for t in tags)
shuffled_data = sample(encoded_data, len(encoded_data))
strat_labels = [1 if contains_entity(x[2].tolist()) else 0 for x in shuffled_data]

train_idx, val_idx = train_test_split(
    range(len(shuffled_data)),
    test_size=0.2,
    random_state=42,
    stratify=strat_labels
)

train_data = [shuffled_data[i] for i in train_idx]
val_data = [shuffled_data[i] for i in val_idx]

# Verification
print("\n=== Data Split Verification ===")
print(f"Total samples: {len(shuffled_data)}")
print(f"Training samples: {len(train_data)} ({len(train_data)/len(shuffled_data):.1%})")
print(f"Validation samples: {len(val_data)} ({len(val_data)/len(shuffled_data):.1%})")

train_ids = {tuple(x[0].tolist()) for x in train_data}
val_ids = {tuple(x[0].tolist()) for x in val_data}
overlap = train_ids & val_ids
#print(f"\nOverlapping samples: {len(overlap)}")
if overlap:
    print("First overlap example:", next(iter(overlap))[:10])
else:
    print("No overlap detected between train and validation sets")

# Dataset and Model
class NERDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]
    def collate_fn(self, batch):
        ids, masks, tags = zip(*batch)
        return (
            torch.stack(ids).to(device),
            torch.stack(masks).to(device),
            torch.stack(tags).to(device),
            torch.tensor([sum(m.tolist()) for m in masks]).to(device)
        )

class BioBERT_BiGRU(nn.Module):
    def __init__(self, tagset_size, hidden_dim=128):
        super().__init__()
        self.bert = biobert
        self.gru = nn.GRU(768, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim*2, tagset_size)

    def forward(self, input_ids, attention_mask, lengths):
        with torch.no_grad():
            outputs = self.bert(input_ids, attention_mask)
        packed = nn.utils.rnn.pack_padded_sequence(outputs.last_hidden_state, lengths.cpu(), batch_first=True)
        gru_out, _ = self.gru(packed)
        unpacked, _ = nn.utils.rnn.pad_packed_sequence(gru_out, batch_first=True)
        return self.classifier(unpacked)

# Initialize
model = BioBERT_BiGRU(len(tag2idx)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([0.7 if idx2tag[i] == 'O' else
                        4.0 if idx2tag[i].startswith('B-') else
                        3.0 for i in range(len(tag2idx))]).to(device),
    ignore_index=pad_tag_id
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=1, factor=0.5)

# Data loaders
train_dataset = NERDataset(train_data)
val_dataset = NERDataset(val_data)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_dataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_dataset.collate_fn)

Processing data: 100%|██████████| 97/97 [00:08<00:00, 11.12it/s]



=== Chunking Stats ===
Total chunks generated: 4895
Average chunk size: 128.0 tokens

=== Deduplication Check ===
After final deduplication: 3906 chunks

=== Data Split Verification ===
Total samples: 3906
Training samples: 3124 (80.0%)
Validation samples: 782 (20.0%)
No overlap detected between train and validation sets


In [80]:
# Training
# Initialize GradScaler (using the correct initialization)
scaler = GradScaler() if torch.cuda.is_available() else None

for epoch in range(15):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for input_ids, masks, tags, lengths in tqdm(train_loader, desc=f"Epoch {epoch+1}/15"):
        optimizer.zero_grad()

        # Handle autocast based on PyTorch version
        if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
            # New PyTorch version (1.10+)
            with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
                # Sort batch by length
                lengths_sorted, sort_idx = lengths.sort(descending=True)
                input_ids_sorted = input_ids[sort_idx]
                masks_sorted = masks[sort_idx]
                tags_sorted = tags[sort_idx]

                logits = model(input_ids_sorted, masks_sorted, lengths_sorted)
                B, L, C = logits.shape
                loss = criterion(logits.view(B * L, C), tags_sorted.view(B * L))
        else:
            # Older PyTorch version
            with autocast(enabled=torch.cuda.is_available()):
                # Sort batch by length
                lengths_sorted, sort_idx = lengths.sort(descending=True)
                input_ids_sorted = input_ids[sort_idx]
                masks_sorted = masks[sort_idx]
                tags_sorted = tags[sort_idx]

                logits = model(input_ids_sorted, masks_sorted, lengths_sorted)
                B, L, C = logits.shape
                loss = criterion(logits.view(B * L, C), tags_sorted.view(B * L))

        # Gradient scaling
        if torch.cuda.is_available():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

        # Convert back to original order for metrics
        _, unsort_idx = sort_idx.sort()
        preds = torch.argmax(logits, dim=-1)[unsort_idx]
        mask = tags != pad_tag_id
        all_preds.extend(preds[mask].cpu().numpy())
        all_labels.extend(tags[mask].cpu().numpy())

    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)

    # Metrics calculation (unchanged from your original code)
    pred_tags = [idx2tag[p] for p in all_preds]
    true_tags = [idx2tag[t] for t in all_labels]
    true_binary = np.array([1 if (t.startswith('B-') or t.startswith('I-')) else 0 for t in true_tags])
    pred_binary = np.array([1 if (p.startswith('B-') or p.startswith('I-')) else 0 for p in pred_tags])

    precision = precision_score(true_binary, pred_binary, zero_division=0)
    recall = recall_score(true_binary, pred_binary, zero_division=0)
    f1 = f1_score(true_binary, pred_binary, zero_division=0)

    #print(f"Epoch {epoch+1} — Loss: {avg_loss:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    # Validation loop
    model.eval()
    val_preds = []
    val_labels = []
    val_loss = 0

    with torch.no_grad():
        for input_ids, masks, tags, lengths in val_loader:
            lengths_sorted, sort_idx = lengths.sort(descending=True)
            input_ids_sorted = input_ids[sort_idx]
            masks_sorted = masks[sort_idx]
            tags_sorted = tags[sort_idx]

            logits = model(input_ids_sorted, masks_sorted, lengths_sorted)
            B, L, C = logits.shape
            loss = criterion(logits.view(B * L, C), tags_sorted.view(B * L))
            val_loss += loss.item()

            _, unsort_idx = sort_idx.sort()
            preds = torch.argmax(logits, dim=-1)[unsort_idx]
            mask = tags != pad_tag_id
            val_preds.extend(preds[mask].cpu().numpy())
            val_labels.extend(tags[mask].cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    val_pred_tags = [idx2tag[p] for p in val_preds]
    val_true_tags = [idx2tag[t] for t in val_labels]
    val_true_binary = np.array([1 if (t.startswith('B-') or t.startswith('I-')) else 0 for t in val_true_tags])
    val_pred_binary = np.array([1 if (p.startswith('B-') or p.startswith('I-')) else 0 for p in val_pred_tags])

    val_precision = precision_score(val_true_binary, val_pred_binary, zero_division=0)
    val_recall = recall_score(val_true_binary, val_pred_binary, zero_division=0)
    val_f1 = f1_score(val_true_binary, val_pred_binary, zero_division=0)


<ipython-input-80-072621a4e1ba>:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if torch.cuda.is_available() else None
Epoch 15/15: 100%|██████████| 98/98 [00:06<00:00, 16.21it/s]


In [81]:
print(f"Validation — Loss: {avg_val_loss:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

Validation — Loss: 0.2903, Precision: 0.9086, Recall: 0.8621, F1: 0.8847


In [82]:
def predict_biobert(text, model, tokenizer, idx2tag, standalone_entity_map=None, stopwords=None):
    model.eval()

    # Step 1: Basic text cleaning and tokenization
    words = [word for word in text.split() if word not in '.,;!?()']  # Filter out common punctuation

    # Step 2: Tokenize with BioBERT
    encoded = tokenizer(words,
                       is_split_into_words=True,
                       return_tensors="pt",
                       padding="max_length",
                       truncation=True,
                       max_length=512)

    # Step 3: Move tensors to device
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    lengths = torch.tensor([attention_mask.sum().item()]).to(device)

    # Step 4: Get model predictions
    with torch.no_grad():
        logits = model(input_ids, attention_mask, lengths)
        preds = torch.argmax(logits, dim=-1).squeeze(0).tolist()

    # Step 5: Align predictions with original words
    word_ids = encoded.word_ids(batch_index=0)
    results = []
    prev_tag = "O"  # Initialize previous tag

    for i, word_idx in enumerate(word_ids):
        if word_idx is not None and (i == 0 or word_idx != word_ids[i-1]):
            token = words[word_idx]
            predicted_tag = idx2tag[preds[i]]

            # Simplified IOB conversion (like first snippet)
            if predicted_tag == "O":
                results.append((token, "O"))
                prev_tag = "O"
            else:
                # Extract base tag without IOB prefix if present
                base_tag = predicted_tag[2:] if predicted_tag.startswith(("B-", "I-")) else predicted_tag

                # Apply IOB rules
                if prev_tag.endswith(base_tag):
                    iob_tag = "I-" + base_tag
                else:
                    iob_tag = "B-" + base_tag

                results.append((token, iob_tag))
                prev_tag = iob_tag

    return results

# Example usage
sample_text = (
    "Wnt signalling the kidney development and Bone morphogenetic protein complex gene drosophila Armadillo modulate morphogenetic abstract Pygo2 homozygous mutants, with exception died shortly birth "
    "Pygo1 and Pygo2 roles in Wnt signaling in mammalian kidney development Abstract Background The pygopus gene of Drosophila encodes an essential component of the Armadillo (β-catenin) transcription factor complex of canonical Wnt signaling To better understand the functions of Pygopus-mediated canonical Wnt signaling in kidney development targeted mutations were made in the two mammalian orthologs, Pygo1 and Pygo2. Results Each mutation deleted >80% of the coding sequence, including the critical PHD domain and almost certainly resulted in null function. Pygo2 homozygous mutants, with rare exception died shortly after birth with a phenotype including lens agenesis, growth retardation, altered kidney development and in some cases exencephaly and cleft palate "
    "Complex trait analysis of the mouse striatum: independent QTLs modulate volume and neuron number Abstract Background The striatum plays a pivotal role in modulating motor activity"
    "Genetic Analysis of the Roles of BMP2, BMP4, and BMP7 in Limb Patterning and Skeletogenesis Abstract Bone morphogenetic protein (BMP) family members, including BMP2, BMP4, and BMP7 are expressed throughout limb development BMPs have been implicated in early limb patterning as well as in the process of skeletogenesis However, due to complications associated with early embryonic lethality, particularly for Bmp2 and Bmp4, and with functional redundancy among BMP molecules, it has been difficult to decipher the specific roles of these BMP molecules during different stages of limb development"
)
results = predict_biobert(sample_text, model, tokenizer, idx2tag, standalone_entity_map, stopwords)

print(f"\n{'Token':<15} {'Tag'}")
print("=" * 30)
for token, tag in results:
    print(f"{token:<15} {tag}")


Token           Tag
Wnt             B-GO:0016055
signalling      B-GO:0060070
the             O
kidney          B-GO:0001822
development     I-GO:0001822
and             O
Bone            B-GO:0060349
morphogenetic   I-GO:0060349
protein         O
complex         O
gene            O
drosophila      O
Armadillo       O
modulate        B-GO:0065007
morphogenetic   B-GO:0009653
abstract        O
Pygo2           O
homozygous      O
mutants,        O
with            O
exception       O
died            B-GO:0016265
shortly         O
birth           B-GO:0007567
Pygo1           O
and             O
Pygo2           O
roles           O
in              O
Wnt             B-GO:0016055
signaling       I-GO:0016055
in              O
mammalian       O
kidney          B-GO:0001822
development     I-GO:0001822
Abstract        O
Background      O
The             O
pygopus         O
gene            O
of              O
Drosophila      O
encodes         O
an              O
essential       O
component      